In [1]:
from google import genai
from google.genai import types
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

# Load API key
load_dotenv('../.env')
api_key = os.getenv('GEMINI_API_KEY')

# Configure Gemini
client = genai.Client(api_key=api_key)

# Connect to PostgreSQL
engine = create_engine('postgresql://paulamipaul@localhost:5432/paulamipaul')

print("✓ Gemini configured (google-genai)")
print("✓ PostgreSQL connected")
print("✓ Ready to build AI agent")

✓ Gemini configured (google-genai)
✓ PostgreSQL connected
✓ Ready to build AI agent


In [5]:
# Define your database schema so the AI knows what tables exist
DB_SCHEMA = """
Tables in the retail database:

1. master_orders - main table with all order details
   columns: order_id, customer_id, order_status, order_purchase_timestamp,
            order_delivered_customer_date, delivery_delay_days,
            processing_time_days, product_id, seller_id, price, freight_value,
            product_category_name_english, product_weight_g,
            customer_city, customer_state, seller_city, seller_state,
            review_score, review_comment_message, payment_value

2. orders_clean - order level data with timestamps
3. products_clean - product details and categories  
4. customers - customer location data
5. sellers - seller location data
6. reviews_clean - customer reviews and scores
7. payments - payment details
8. order_items - individual items per order
"""

def ask_retail_agent(question):
    # Step 1: Ask Gemini to write a SQL query
    prompt = f"""
You are a retail data analyst. Given this PostgreSQL database schema:
{DB_SCHEMA}

Write a single PostgreSQL SQL query to answer this question:
"{question}"

Rules:
- Return ONLY the SQL query, no explanation
- Use master_orders as the main table when possible
- Always use LIMIT 20 unless asked for all data
- Use proper PostgreSQL syntax
"""
    
    sql_response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    
    # Clean the SQL
    sql = sql_response.text.strip()
    sql = sql.replace('```sql', '').replace('```', '').strip()
    
    print(f"Question: {question}")
    print(f"\nGenerated SQL:\n{sql}\n")
    
    # Step 2: Run the SQL on PostgreSQL
    try:
        with engine.connect() as conn:
            result = pd.read_sql(text(sql), conn)
        print(f"Results ({len(result)} rows):")
        return result
    except Exception as e:
        print(f"SQL Error: {e}")
        return None

print("✓ AI agent function ready")

✓ AI agent function ready


In [6]:
# Test your agent with a real business question
result = ask_retail_agent("Which product categories have the highest average delivery delay?")
result

Question: Which product categories have the highest average delivery delay?

Generated SQL:
SELECT
  product_category_name_english,
  AVG(delivery_delay_days) AS avg_delivery_delay
FROM master_orders
WHERE
  delivery_delay_days IS NOT NULL
GROUP BY
  product_category_name_english
ORDER BY
  avg_delivery_delay DESC
LIMIT 20;

Results (20 rows):


,product_category_name_english,avg_delivery_delay
0,arts_and_craftmanship,-6.791667
1,furniture_mattress_and_upholstery,-7.162162
2,home_comfort_2,-8.433333
3,home_confort,-9.858796
4,food,-9.867735
5,audio,-10.140496
6,fashion_underwear_beach,-10.929134
7,electronics,-11.139560
8,books_imported,-11.192982
9,cine_photo,-11.211268


In [7]:
result2 = ask_retail_agent("Which states have the most orders and what is their average review score?")
result2

Question: Which states have the most orders and what is their average review score?

Generated SQL:
SELECT
  customer_state,
  COUNT(DISTINCT order_id) AS total_orders,
  AVG(review_score) AS average_review_score
FROM
  master_orders
GROUP BY
  customer_state
ORDER BY
  total_orders DESC
LIMIT 20;

Results (20 rows):


,customer_state,total_orders,average_review_score
0,SP,41746,4.108680
1,RJ,12852,3.793639
2,MG,11635,4.070536
3,RS,5466,4.042249
4,PR,5045,4.087671
5,SC,3637,3.989237
6,BA,3380,3.801954
7,DF,2140,3.991354
8,ES,2033,3.982628
9,GO,2020,3.981537


In [8]:
result3 = ask_retail_agent("Who are the top 10 sellers by total revenue?")
result3

Question: Who are the top 10 sellers by total revenue?

Generated SQL:
SELECT
  seller_id,
  SUM(price + freight_value) AS total_revenue
FROM master_orders
GROUP BY
  seller_id
ORDER BY
  total_revenue DESC
LIMIT 10;

Results (10 rows):


,seller_id,total_revenue
0,None,NaN
1,4869f7a5dfa277a7dca6462dcf3b52b2,249640.70
2,7c67e1448b00f6e969d365cea6b010ab,241374.82
3,4a3ca9315b744ce9f8e9374361493884,238440.31
4,53243585a1d6dc2643021fd1853d8905,235856.68
5,fa1c13f2614d7b5c4749cbc52fecda94,204084.73
6,da8622b14eb17ae2831f4ac5b9dab84a,188062.51
7,7e93a43ef30c4f03f38b393420bc753a,182754.05
8,1025f0e2d44d7041d6cf58b6550e0bfa,174710.82
9,7a67c85e85bb2ce8582c35f2203ad736,163278.78


In [9]:
result4 = ask_retail_agent("What is the monthly revenue trend over time?")
result4

Question: What is the monthly revenue trend over time?

Generated SQL:
SELECT
  TO_CHAR(order_purchase_timestamp, 'YYYY-MM') AS sales_month,
  SUM(payment_value) AS monthly_revenue
FROM master_orders
GROUP BY
  sales_month
ORDER BY
  sales_month
LIMIT 20;

Results (20 rows):


,sales_month,monthly_revenue
0,2016-09,388.47
1,2016-10,76559.05
2,2016-12,19.62
3,2017-01,190806.27
4,2017-02,351848.13
5,2017-03,547769.84
6,2017-04,512126.52
7,2017-05,737425.31
8,2017-06,613777.41
9,2017-07,749242.84
